# Basic cleaning and exploratory data analysis (EDA)

## Importing libraries

In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Importing stock data from yfinance

In [3]:
tickers = ['HDFCBANK.NS', 'ICICIBANK.NS', '^NSEBANK']
data = yf.download(tickers, period = "10y", auto_adjust=True)
# auto_adjust=True for handling stock splits and dividends in the source data.
# Creating a close price only dataframe for the calculations later on.
close = data['Close']

[*********************100%***********************]  3 of 3 completed


## Quick view of the data

In [4]:
print("Main data ->")
print(data.head())
print("Closed price data ->")
print(close.head())

Main data ->
Price            Close                                   High               \
Ticker     HDFCBANK.NS ICICIBANK.NS      ^NSEBANK HDFCBANK.NS ICICIBANK.NS   
Date                                                                         
2016-03-21  240.374512   199.372635  15925.914062  240.776822   200.009342   
2016-03-22  242.271072   198.311447  15935.814453  242.903274   199.160381   
2016-03-23  241.236603   198.820786  15887.565430  242.087218   199.499941   
2016-03-28  240.811264   191.392609  15604.718750  242.075667   198.651005   
2016-03-29  242.259613   189.652298  15666.068359  243.569983   192.666013   

Price                            Low                                   Open  \
Ticker          ^NSEBANK HDFCBANK.NS ICICIBANK.NS      ^NSEBANK HDFCBANK.NS   
Date                                                                          
2016-03-21  15945.614028  237.305466   196.486250  15735.366478  237.477885   
2016-03-22  15967.663692  239.247995   195.764

## Checking datatypes of the columns

In [5]:
print(data.info())
print(close.info())

<class 'pandas.DataFrame'>
DatetimeIndex: 2470 entries, 2016-03-21 to 2026-03-19
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   (Close, HDFCBANK.NS)    2470 non-null   float64
 1   (Close, ICICIBANK.NS)   2470 non-null   float64
 2   (Close, ^NSEBANK)       2466 non-null   float64
 3   (High, HDFCBANK.NS)     2470 non-null   float64
 4   (High, ICICIBANK.NS)    2470 non-null   float64
 5   (High, ^NSEBANK)        2466 non-null   float64
 6   (Low, HDFCBANK.NS)      2470 non-null   float64
 7   (Low, ICICIBANK.NS)     2470 non-null   float64
 8   (Low, ^NSEBANK)         2466 non-null   float64
 9   (Open, HDFCBANK.NS)     2470 non-null   float64
 10  (Open, ICICIBANK.NS)    2470 non-null   float64
 11  (Open, ^NSEBANK)        2466 non-null   float64
 12  (Volume, HDFCBANK.NS)   2470 non-null   int64  
 13  (Volume, ICICIBANK.NS)  2470 non-null   int64  
 14  (Volume, ^NSEBANK)      2466 non-

## Checking if any of the columns has any nan values

In [6]:
print("nan values in the main data ->")
print(data.isna().sum())
print("nan values in the close price data ->")
print(close.isna().sum())

nan values in the main data ->
Price   Ticker      
Close   HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        4
High    HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        4
Low     HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        4
Open    HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        4
Volume  HDFCBANK.NS     0
        ICICIBANK.NS    0
        ^NSEBANK        4
dtype: int64
nan values in the close price data ->
Ticker
HDFCBANK.NS     0
ICICIBANK.NS    0
^NSEBANK        4
dtype: int64


# Seeing what are these nan valued columns

In [7]:
nan_value = data[data.isna().any(axis=1)]
nan_date = data[data.isna().any(axis=1)].index
print(nan_value)

Price            Close                              High               \
Ticker     HDFCBANK.NS ICICIBANK.NS ^NSEBANK HDFCBANK.NS ICICIBANK.NS   
Date                                                                    
2019-10-27  586.159607   454.142914      NaN  588.148710   457.963317   
2020-11-14  648.673218   470.923645      NaN  651.159554   473.776860   
2025-02-01  834.199097  1245.936279      NaN  845.077083  1251.691796   
2026-01-15  925.450012  1418.400024      NaN  925.450012  1418.400024   

Price                       Low                              Open  \
Ticker     ^NSEBANK HDFCBANK.NS ICICIBANK.NS ^NSEBANK HDFCBANK.NS   
Date                                                                
2019-10-27      NaN  584.880943   449.742222      NaN  587.225180   
2020-11-14      NaN  646.542073   468.747473      NaN  647.773447   
2025-02-01      NaN  826.848504  1224.352819      NaN  837.430443   
2026-01-15      NaN  925.450012  1418.400024      NaN  925.450012   

Pric

## Why nan?
These days the trading of the instrument did not happen so it is showing nan, the volume is also 0 for a couple of the banks so will check them too. Will use forward fill (using last known value) instead of backward fill (using next known value.)

Since it is volume, we will check if bank nifty has 0 volume, if so will replace the volumes in the others if there are any like on 2026-01-15 as it is most likely an error as bank nifty always has volume unless it is a holiday. What to do
- Will remove the 0 volume days entirely as they are holdiays to being with and will skew the data.
- Keep them but filter later on or create a copy data frame to do the stuff later on.

In [8]:
zero_volume_date = data[(data['Volume']==0).any(axis=1)].index
print(zero_volume_date)

DatetimeIndex(['2016-11-10', '2016-12-16', '2016-12-19', '2016-12-20',
               '2016-12-21', '2016-12-22', '2016-12-23', '2016-12-27',
               '2016-12-28', '2016-12-29',
               ...
               '2021-08-09', '2022-08-16', '2023-03-22', '2023-08-16',
               '2024-07-01', '2024-07-02', '2024-07-03', '2025-03-18',
               '2026-01-15', '2026-03-19'],
              dtype='datetime64[s]', name='Date', length=920, freq=None)


In [9]:
zero_volume_date_nse = data[(data['Volume']['^NSEBANK']==0)].index
print(zero_volume_date)

DatetimeIndex(['2016-11-10', '2016-12-16', '2016-12-19', '2016-12-20',
               '2016-12-21', '2016-12-22', '2016-12-23', '2016-12-27',
               '2016-12-28', '2016-12-29',
               ...
               '2021-08-09', '2022-08-16', '2023-03-22', '2023-08-16',
               '2024-07-01', '2024-07-02', '2024-07-03', '2025-03-18',
               '2026-01-15', '2026-03-19'],
              dtype='datetime64[s]', name='Date', length=920, freq=None)


In [10]:
difference = list(set(zero_volume_date) - set(zero_volume_date_nse)) # Convert to set for using `-` then use list to convert back to list.
print(difference)

[Timestamp('2025-03-18 00:00:00'), Timestamp('2026-01-15 00:00:00')]


In [11]:
for i in difference:
    print(data[data.index == i])


Price            Close                              High               \
Ticker     HDFCBANK.NS ICICIBANK.NS ^NSEBANK HDFCBANK.NS ICICIBANK.NS   
Date                                                                    
2025-03-18  843.794495  1259.283325  49314.5  843.794495  1259.283325   

Price                            Low                                   Open  \
Ticker          ^NSEBANK HDFCBANK.NS ICICIBANK.NS      ^NSEBANK HDFCBANK.NS   
Date                                                                          
2025-03-18  49400.300781  843.794495  1259.283325  48629.449219  843.794495   

Price                                      Volume                         
Ticker     ICICIBANK.NS      ^NSEBANK HDFCBANK.NS ICICIBANK.NS  ^NSEBANK  
Date                                                                      
2025-03-18  1259.283325  48792.898438           0            0  146600.0  
Price            Close                              High               \
Ticker     HDFCBA

# Removing rows with this logic
- Check nse bank if it is 0 or nan
- On the days of the previous one, check if either bank has 0 or nan values
- If either bank has 0 or nan, drop, if none have then check nse bank.

In [16]:
nse_miss = (data['Volume']['^NSEBANK'] == 0) | (data['Volume']['^NSEBANK'].isna())
remove_nse_miss = data[~nse_miss].copy()

In [ ]:
print(remove_nse_miss[(remove_nse_miss['Volume']['^NSEBANK'] == 0) | (remove_nse_miss['Volume']['^NSEBANK'].isna())])

Empty DataFrame
Columns: [(Close, HDFCBANK.NS), (Close, ICICIBANK.NS), (Close, ^NSEBANK), (High, HDFCBANK.NS), (High, ICICIBANK.NS), (High, ^NSEBANK), (Low, HDFCBANK.NS), (Low, ICICIBANK.NS), (Low, ^NSEBANK), (Open, HDFCBANK.NS), (Open, ICICIBANK.NS), (Open, ^NSEBANK), (Volume, HDFCBANK.NS), (Volume, ICICIBANK.NS), (Volume, ^NSEBANK)]
Index: []


In [23]:
print("HDFC")
print((remove_nse_miss[(remove_nse_miss['Volume']['HDFCBANK.NS'] == 0) | (remove_nse_miss['Volume']['HDFCBANK.NS'].isna())]).count())
print("ICICI")
print((remove_nse_miss[(remove_nse_miss['Volume']['ICICIBANK.NS'] == 0) | (remove_nse_miss['Volume']['ICICIBANK.NS'].isna())]).count())

HDFC
Price   Ticker      
Close   HDFCBANK.NS     1
        ICICIBANK.NS    1
        ^NSEBANK        1
High    HDFCBANK.NS     1
        ICICIBANK.NS    1
        ^NSEBANK        1
Low     HDFCBANK.NS     1
        ICICIBANK.NS    1
        ^NSEBANK        1
Open    HDFCBANK.NS     1
        ICICIBANK.NS    1
        ^NSEBANK        1
Volume  HDFCBANK.NS     1
        ICICIBANK.NS    1
        ^NSEBANK        1
dtype: int64
ICICI
Price   Ticker      
Close   HDFCBANK.NS     1
        ICICIBANK.NS    1
        ^NSEBANK        1
High    HDFCBANK.NS     1
        ICICIBANK.NS    1
        ^NSEBANK        1
Low     HDFCBANK.NS     1
        ICICIBANK.NS    1
        ^NSEBANK        1
Open    HDFCBANK.NS     1
        ICICIBANK.NS    1
        ^NSEBANK        1
Volume  HDFCBANK.NS     1
        ICICIBANK.NS    1
        ^NSEBANK        1
dtype: int64


In [25]:
print("HDFC")
print((remove_nse_miss[(remove_nse_miss['Volume']['HDFCBANK.NS'] == 0) | (remove_nse_miss['Volume']['HDFCBANK.NS'].isna())].index))
print("ICICI")
print((remove_nse_miss[(remove_nse_miss['Volume']['ICICIBANK.NS'] == 0) | (remove_nse_miss['Volume']['ICICIBANK.NS'].isna())].index))

HDFC
DatetimeIndex(['2025-03-18'], dtype='datetime64[s]', name='Date', freq=None)
ICICI
DatetimeIndex(['2025-03-18'], dtype='datetime64[s]', name='Date', freq=None)


In [27]:
print(remove_nse_miss[remove_nse_miss.index == '2025-03-18'])

Price            Close                              High               \
Ticker     HDFCBANK.NS ICICIBANK.NS ^NSEBANK HDFCBANK.NS ICICIBANK.NS   
Date                                                                    
2025-03-18  843.794495  1259.283325  49314.5  843.794495  1259.283325   

Price                            Low                                   Open  \
Ticker          ^NSEBANK HDFCBANK.NS ICICIBANK.NS      ^NSEBANK HDFCBANK.NS   
Date                                                                          
2025-03-18  49400.300781  843.794495  1259.283325  48629.449219  843.794495   

Price                                      Volume                         
Ticker     ICICIBANK.NS      ^NSEBANK HDFCBANK.NS ICICIBANK.NS  ^NSEBANK  
Date                                                                      
2025-03-18  1259.283325  48792.898438           0            0  146600.0  
